## Text Chunking - Recursive Character Text Splitting and Semantic Similarity Splitting

#### Load in Python Libraries

In [1]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceBgeEmbeddings
from tqdm import tqdm
import pandas as pd
from rich import print
import torch
import gc

import pickle
import itertools

from collections import ChainMap

import mlflow

#### Set mlFlow Experiment

In [2]:
mlflow.set_experiment("adeptID")
mlflow.start_run(run_name = "langchain-text-chunking")

<ActiveRun: >

#### Helper Functions

In [16]:
def count_words(text):

    words = text.split()

    return len(words)

def batch_dict(d, batch_size):
    
    keys = list(d.keys())

    for i in range(0, len(keys), batch_size):

        yield {key: d[key] for key in keys[i:i+batch_size]}

def process_texts(input_dict, text_splitter):
    
    output_dict = {}

    for key, text in tqdm(input_dict.items()):

        chunks = text_splitter.create_documents([text])

        split_text = [chunk.page_content for chunk in chunks]
        
        output_dict[key] = split_text
        
        torch.cuda.empty_cache()
        
        gc.collect()

    return output_dict

def filter_dict_by_value_length(data_dict, threshold):
    
    filtered_keys = [key for key, value in data_dict.items() if len(value) > threshold]

    remaining_dict = {key: value for key, value in data_dict.items() if len(value) <= threshold}

    return filtered_keys, remaining_dict

def create_extended_long_postings(filtered_keys, df):

    long_posting_list = list(itertools.chain(filtered_keys))

    df2 = df[df['id'].isin(long_posting_list)].copy()

    text_dict_long_extend = {key: value for key, value in zip(df2['id'], df2['body'])}

    #extended_long_postings = ChainMap(text_dict_long, text_dict_long_extend)

    return text_dict_long_extend

#### Load in Unclassified Data Collected from LightCast Dataset

In [6]:
df = pd.read_csv(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\unclassified_postings.csv", index_col =0)
df['word_count'] = df['body'].apply(count_words)
df = df[df['word_count'] > 50].reset_index(drop = True)
mlflow.log_params({'max_word_count': max(df['word_count']),'min_word_count': min(df['word_count'])})

#### Load in Semantic Text Splitting Model and Recursive Text Splitter 

In [7]:
#Semantic Chunking Model
model_name = "nomic-ai/nomic-embed-text-v1.5"
#Set to device to cpu or cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_kwargs = {'device': device, 'trust_remote_code': True}
#normalize to embed faster 
encode_kwargs = {'normalize_embeddings': True}

#Use LangChain to do text embeddings
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)
mlflow.log_params({'model': model_name,'device': device})

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
<All keys matched successfully>


In [8]:
#Semantic Text Spliter
semantic_text_splitter = SemanticChunker(embeddings=embeddings,
                                 breakpoint_threshold_type="percentile",
                                 breakpoint_threshold_amount=5,
                                 )

#Recursive Text Splitter
recursive_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

In [9]:
#Set to desired Batch Size and Token Length (normally set at 20,000 set to 1000 for example purposes)
batch_size = 1000
max_token_length = 8000
mlflow.log_params({'batch_size': batch_size, 'max_token_length': max_token_length})

#### Separate long job postings from short job posting with threshold of 1500

In [10]:
df_short = df[df['word_count'] <= 1500].reset_index(drop = True)
df_long = df[df['word_count'] > 1500].reset_index(drop = True)
text_dict_short = {key:value for key, value in zip(df_short['id'], df_short['body'])}
text_dict_long = {key:value for key, value in zip(df_long['id'], df_long['body'])}

In [11]:
batches = {}

for i, batch in enumerate(batch_dict(text_dict_short, batch_size)):
    batches[f'batch_{i+1}'] = batch
mlflow.log_param('number_of_batches', len(batches))

216

In [12]:
batch_number = 11

#### Add token lengths above 8000 to Recursive Text Splitting 

In [17]:
#filtered keys goes to create_extend_long_posting function 
filtered_keys, remaining_dict = filter_dict_by_value_length(batches[f'batch_{batch_number}'], max_token_length)
#if want to combine all for recursive 
text_dict_long_extend = create_extended_long_postings(filtered_keys, df)
mlflow.log_params({'batch_number': batch_number, 'semantic_chunking_len': len(remaining_dict), 'recursive_chunking_len': len(text_dict_long_extend)})

#### Semantic Text Splitting

In [19]:
semantic_text_dict = process_texts(remaining_dict, semantic_text_splitter)

100%|██████████| 974/974 [08:05<00:00,  2.01it/s]


#### Recursive Character Text Splitter

In [18]:
recursive_text_dict = process_texts(text_dict_long_extend,recursive_text_splitter)

100%|██████████| 26/26 [00:03<00:00,  7.10it/s]


#### Combine Dictionaries

In [20]:
batch_dict = semantic_text_dict | recursive_text_dict

#### Save Batch as Pickle 

In [21]:
with open(f'job_posting_chunk_batch{batch_number}.pickle', 'wb') as handle:
    pickle.dump(batch_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [22]:
mlflow.end_run()